## Задание 1

Реализуйте класс с полностью инкапсулированным состоянием, используя name mangling и property, обеспечив валидацию при изменении атрибутов и демонстрируя, как Python скрывает "приватные" данные.

In [2]:
class Encapsulated:
    def __init__(self, value):
        self.__value = None
        self.value = value

    @property
    def value(self):
        """Getter for the private __value."""
        return self.__value

    @value.setter
    def value(self, new_value):
        """Setter with validation: must be a positive integer."""
        if not isinstance(new_value, int):
            raise TypeError("Value must be an integer")
        if new_value <= 0:
            raise ValueError("Value must be positive")
        self.__value = new_value

obj = Encapsulated(10)
print(obj.value)
print(obj._Encapsulated__value)

10
10


## Задание 2

Создайте иерархию классов с абстрактным базовым классом (ABC) и абстрактными методами, демонстрируя использование модуля abc. Реализуйте интерфейсы и их проверку через isinstance и issubclass.

In [5]:
from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass

    @abstractmethod
    def move(self):
        pass

class Dog(Animal):
    def sound(self):
        return "Woof!"
    
    def move(self):
        return "Running on four legs"

class Cat(Animal):
    def sound(self):
        return "Meow!"
    
    def move(self):
        return "Walking stealthily"

# Instantiate
dog = Dog()
cat = Cat()

print(dog.sound())
print(cat.move())

# Check using isinstance
print('-' * 15)
print(isinstance(dog, Animal))   # True
print(isinstance(cat, Animal))   # True
print(isinstance(dog, Dog))      # True
print(isinstance(cat, Cat))      # True

# Check using issubclass
print('-' * 15)
print(issubclass(Dog, Animal))   # True
print(issubclass(Cat, Animal))   # True
print(issubclass(Animal, ABC))   # True (ABC is a metaclass)

Woof!
Walking stealthily
---------------
True
True
True
True
---------------
True
True
True


## Задание 3

Реализуйте дженерик-функцию (duck typing) с использованием протоколов (PEP 544) и типовых подсказок, чтобы показать полиморфизм без наследования.

In [6]:
# https://peps.python.org/pep-0544/
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> None:
        ...

class Circle:
    def draw(self) -> None:
        print("Drawing a circle")

class Square:
    def draw(self) -> None:
        print("Drawing a square")

def paint(obj: Drawable) -> None:
    obj.draw()

if __name__ == "__main__":
    paint(Circle())   # Drawing a circle
    paint(Square())   # Drawing a square

Drawing a circle
Drawing a square


## Задание 4

Опишите и продемонстрируйте работу метода getattribute в отличие от getattr, реализуйте логирование всех обращений к атрибутам объекта, исследуйте возможные зацикливания.

In [7]:
class Logger:
    def __init__(self):
        self.x = 5

    def __getattribute__(self, name):
        # Called for EVERY attribute access (including from within __getattribute__ itself)
        print(f"[__getattribute__] Accessing attribute: {name}")
        try:
            # Use super() to avoid infinite recursion
            return super().__getattribute__(name)
        except AttributeError:
            # If attribute doesn't exist, __getattr__ will be called (if defined)
            # But we must raise AttributeError to trigger __getattr__
            raise

    def __getattr__(self, name):
        # Called only when attribute is NOT found via normal lookup
        print(f"[__getattr__] Attribute '{name}' not found, returning default")
        return None

if __name__ == "__main__":
    obj = Logger()
    obj.z = 42
    print("Access existing attribute:")
    print(obj.x)

    print("\nAccess missing attribute:")
    print(obj.y)

Access existing attribute:
[__getattribute__] Accessing attribute: x
5

Access missing attribute:
[__getattribute__] Accessing attribute: y
[__getattr__] Attribute 'y' not found, returning default
None


## Задание 5

Создайте класс, который использует метакласс, объясните как метаклассы влияют на создание классов в Python, и реализуйте контролируемое изменение класса через метакласс.

In [8]:
'''
Метакласс Meta наследуется от type и переопределяет __new__.
При создании класса GoodClass (и любого другого, использующего этот метакласс) 
метакласс получает имя класса, кортеж базовых классов и словарь атрибутов (namespace).
Мы преобразуем все методы класса (исключая специальные) к нижнему регистру, 
добавляем атрибут _meta_version и метод meta_info. 
Это демонстрирует, как метаклассы могут контролировать создание классов:
менять имена, добавлять атрибуты, модифицировать поведение.
'''

class Meta(type):
    def __new__(mcs, name, bases, namespace):
        new_namespace = {}
        for key, value in namespace.items():
            if callable(value) and not (key.startswith('__') and key.endswith('__')):
                new_namespace[key.lower()] = value
            else:
                new_namespace[key] = value

        new_namespace['_meta_version'] = '1.0'

        def meta_info(cls):
            return f"Class {cls.__name__} created with metaclass Meta, version {cls._meta_version}"
        new_namespace['meta_info'] = classmethod(meta_info)

        return super().__new__(mcs, name, bases, new_namespace)


class GoodClass(metaclass=Meta):
    def SayHello(self):
        return "Hello!"
    def CalCulAte(self):
        return 42
    class_attr = "some value"


if __name__ == "__main__":
    print("Атрибуты класса:")
    print(f"_meta_version: {GoodClass._meta_version}")
    print(f"meta_info(): {GoodClass.meta_info()}")
    print(f"class_attr: {GoodClass.class_attr}")
    print("Имена методов после преобразования:")
    print(f"sayhello: {hasattr(GoodClass, 'sayhello')}")
    print(f"calculate: {hasattr(GoodClass, 'calculate')}")
    print(f"Исходные имена (в верхнем регистре) удалены:")
    print(f"SayHello: {hasattr(GoodClass, 'SayHello')}")
    print(f"CalCulAte: {hasattr(GoodClass, 'CalCulAte')}")

    obj = GoodClass()
    print("\nВызов преобразованных методов:")
    print(obj.sayhello())
    print(obj.calculate())

Атрибуты класса:
_meta_version: 1.0
meta_info(): Class GoodClass created with metaclass Meta, version 1.0
class_attr: some value
Имена методов после преобразования:
sayhello: True
calculate: True
Исходные имена (в верхнем регистре) удалены:
SayHello: False
CalCulAte: False

Вызов преобразованных методов:
Hello!
42


## Задание 6

Реализуйте класс с дескрипторами данных и неданных, объясните разницу между ними и механизм вызова методов get, set, delete, особенно при наследовании.

In [9]:
'''
Дескрипторы данных (data descriptors) реализуют методы __set__ и/или __delete__.
Они имеют высший приоритет при доступе к атрибуту: даже если в __dict__ экземпляра есть атрибут с таким же именем,
дескриптор данных перехватывает обращение. Это позволяет контролировать запись и удаление.
Дескрипторы не-данных (non-data descriptors) реализуют только __get__.
Они имеют низший приоритет: если в __dict__ экземпляра есть атрибут, он будет использован вместо дескриптора.
При наследовании дескрипторы наследуются, и логика приоритетов сохраняется.
'''

class DataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        # Получаем значение из приватного словаря экземпляра
        return instance.__dict__.get('_data_desc_val', None)

    def __set__(self, instance, value):
        # Сохраняем значение в словарь экземпляра с особым ключом
        instance.__dict__['_data_desc_val'] = value

    def __delete__(self, instance):
        # Удаляем из словаря экземпляра
        if '_data_desc_val' in instance.__dict__:
            del instance.__dict__['_data_desc_val']


class NonDataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        # Возвращаем вычисляемое значение, но не сохраняем
        return 'значение из non-data дескриптора'


class MyClass2:
    data_desc = DataDescriptor()
    non_data_desc = NonDataDescriptor()


if __name__ == '__main__':
    obj = MyClass2()

    # Работа с data descriptor: установка значения через дескриптор
    obj.data_desc = 10
    print(obj.data_desc)  # выведет 10

    # У data descriptor есть __set__, поэтому obj.data_desc = 10 вызвал дескриптор,
    # и значение сохранилось в instance.__dict__['_data_desc_val'].
    # При последующем доступе obj.data_desc __get__ вернёт это значение.

    # Работа с non-data descriptor: он возвращает фиксированное значение
    print(obj.non_data_desc)  # выведет строку из дескриптора

    # Теперь установим значение в __dict__ экземпляра для non_data_desc
    obj.non_data_desc = 'переопределённое значение'
    print(obj.non_data_desc)  # выведет 'переопределённое значение', потому что атрибут в __dict__ перекрывает дескриптор

    # У data descriptor такой трюк не сработает: даже если мы запишем в __dict__,
    # дескриптор данных всё равно перехватит доступ:
    obj.__dict__['data_desc'] = 'попытка обхода'
    print(obj.data_desc)  # всё равно выведет 10, так как __get__ data дескриптора возвращает _data_desc_val

    # Демонстрация наследования: дескрипторы наследуются
    class Child(MyClass2):
        pass

    child = Child()
    child.data_desc = 20
    print(child.data_desc)  # 20 – дескриптор из родителя работает

10
значение из non-data дескриптора
переопределённое значение
10
20


## Задание 7

Напишите класс с поддержкой множественного наследования, демонстрирующий работу C3-линеаризации MRO на сложном примере с 3+ уровнями наследования и пересечениями.

In [11]:
class A:
    def method(self):
        print("A.method")

class B(A):
    def method(self):
        print("B.method")
        super().method()

class C(A):
    def method(self):
        print("C.method")
        super().method()

class D(B, C):
    def method(self):
        print("D.method")
        super().method()


d = D()
d.method()
print("\nMRO для D:")
print(D.__mro__)

D.method
B.method
C.method
A.method

MRO для D:
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


## Задание 8

Реализуйте класс с пользовательскими dunder-методами: new, init, call, del, repr, str, и объясните, как Python вызывает их в цикле жизни объекта.

In [12]:

class LifeCycle:
    def __new__(cls, *args, **kwargs):
        print("1. __new__: создание экземпляра")
        instance = super().__new__(cls)
        print("   __new__ возвращает экземпляр")
        return instance

    def __init__(self, name="Аноним"):
        print(f"2. __init__: инициализация с именем '{name}'")
        self.name = name

    def __repr__(self):
        # официальное представление, используется в интерактивной среде
        return f"LifeCycle('{self.name}')"

    def __str__(self):
        # человеко-читаемое представление, используется в print()
        return f"Объект LifeCycle с именем '{self.name}'"

    def __call__(self, *args, **kwargs):
        print(f"3. __call__: объект вызван как функция с аргументами {args}")

    def __del__(self):
        # вызывается при удалении объекта (не гарантируется точное время)
        print("4. __del__: объект уничтожается")


if __name__ == "__main__":
    print("--- Создание объекта ---")
    obj = LifeCycle("Питон")

    print("\n--- Вывод через print() ---")
    print(obj)          # вызывает __str__

    print("\n--- Вызов объекта как функции ---")
    obj(42, "тест")     # вызывает __call__

    print("\n--- Явное удаление ссылки ---")
    del obj             # вызывает __del__ (но не мгновенно, зависит от сборщика)

    print("--- Конец программы ---")

--- Создание объекта ---
1. __new__: создание экземпляра
   __new__ возвращает экземпляр
2. __init__: инициализация с именем 'Питон'

--- Вывод через print() ---
Объект LifeCycle с именем 'Питон'

--- Вызов объекта как функции ---
3. __call__: объект вызван как функция с аргументами (42, 'тест')

--- Явное удаление ссылки ---
4. __del__: объект уничтожается
--- Конец программы ---


## Задание 9

Создайте контекстный менеджер с помощью специальных методов enter и exit, используйте его вместе с классом, в котором присутствуют методы с разграничением прав доступа.

https://habr.com/ru/articles/739326/

In [13]:
'''
Класс Access управляет доступом к секретному значению _secret.
Доступ к свойству secret разрешён только внутри блока with.
Контекстный менеджер реализован через методы __enter__ и __exit__.
'''

class Access:
    def __init__(self):
        self._secret = "Секретные данные"
        self._access_count = 0          # счётчик вложенных контекстов

    @property
    def secret(self):
        """Свойство, доступное только при активном контексте."""
        if self._access_count > 0:
            return self._secret
        else:
            raise PermissionError("Доступ к секрету разрешён только внутри контекста with")

    def __enter__(self):
        """Вход в контекст: увеличиваем счётчик и возвращаем секретное значение."""
        self._access_count += 1
        return self._secret             # значение, которое попадёт в переменную после 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Выход из контекста: уменьшаем счётчик."""
        self._access_count -= 1
        # Если произошло исключение, можно его обработать (но здесь просто пропускаем)
        return False                    # исключение не подавляем

obj = Access()

# Попытка доступа вне контекста
try:
    print(obj.secret)
except PermissionError as e:
    print(f"Ошибка: {e}")

# Использование контекстного менеджера
with obj as secret_value:
    print(f"Внутри контекста: {secret_value}")   # secret_value = obj._secret
    print(f"Доступ через свойство: {obj.secret}")

# После выхода из контекста доступ снова запрещён
try:
    print(obj.secret)
except PermissionError as e:
    print(f"Ошибка: {e}")

Ошибка: Доступ к секрету разрешён только внутри контекста with
Внутри контекста: Секретные данные
Доступ через свойство: Секретные данные
Ошибка: Доступ к секрету разрешён только внутри контекста with


## Задание 10

Создайте класс, объекты которого могут быть отслежены с помощью слабых ссылок (weakref). Реализуйте систему, которая хранит слабые ссылки на все созданные объекты класса и автоматически удаляет их из списка при уничтожении объектов. Продемонстрируйте это поведение, выводя текущее количество живых объектов.

In [ ]:
import weakref
import gc

class Tracked:
    """Класс, объекты которого отслеживаются через слабые ссылки."""
    _instances = weakref.WeakSet()

    def __init__(self, name):
        self.name = name
        self._instances.add(self)
        print(f"Создан {self.name}")

    @classmethod
    def get_instance_count(cls):
        """Возвращает количество живых объектов."""
        return len(cls._instances)

    def __repr__(self):
        return f"Tracked('{self.name}')"

print("Создаём объекты:")
a = Tracked("A")
b = Tracked("B")
c = Tracked("C")

print(f"\nКоличество живых объектов: {Tracked.get_instance_count()}")

print("\nУдаляем ссылку на b (del b):")
del b

gc.collect() # Принудительно собираем мусор, чтобы weakset обновился
print(f"Количество живых объектов: {Tracked.get_instance_count()}")

print("\nУдаляем ссылку на a (del a):")
del a
gc.collect()
print(f"Количество живых объектов: {Tracked.get_instance_count()}")

print("\nОстался только c:")
print(f"Живые объекты: {list(Tracked._instances)}")

Создаём объекты:
Создан A
Создан B
Создан C

Количество живых объектов: 3

Удаляем ссылку на b (del b):
Количество живых объектов: 2

Удаляем ссылку на a (del a):
Количество живых объектов: 1

Остался только c:
Живые объекты: [Tracked('C')]
